# NeuroPath AI — Tremor Severity Classifier (Notebook 02)

**Model:** SVM (RBF) on FFT + statistical features  
**Dataset:** `imu-hand-tremor-parkinsons.csv` (verified structure)  
**Output:** `../models/tremor_classifier.pkl` + scaler + config

### Verified dataset structure
```
Columns: xAcc, yAcc, zAcc, xGyro, yGyro, zGyro, label
Accel units: g  (Z rests at ~0.99)
Gyro units : deg/s
Rows       : 4502 (one time-step per row)
Layout     : 3 contiguous blocks
             rows    0-1499  -> label 1 (mild tremor)
             rows 1500-2999  -> label 2 (moderate tremor)
             rows 3000-4501  -> label 3 (high tremor)
```

### Label meaning (confirmed)
| label | severity |
|-------|----------|
| 1 | mild |
| 2 | moderate |
| 3 | high |

### Why windowing
Each row is a single 50 Hz sample. Tremor is a *frequency* phenomenon,
so we slice each class block into overlapping windows and extract FFT
features per window. This matches exactly what the live app does via
`feature_adapter.prepare_tremor_input()`.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, json, pickle
from scipy.fft import fft, fftfreq

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print('Imports OK')

## 2. Configure paths

In [ ]:
DATA_PATH  = '../data/imu-hand-tremor-parkinsons.csv'
MODELS_DIR = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

# Sampling rate of the dataset AND your glove. Keep them equal.
FS = 50

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError('Put imu-hand-tremor-parkinsons.csv in ../data/')
print('Data found:', os.path.abspath(DATA_PATH))

## 3. Load and verify

In [ ]:
df = pd.read_csv(DATA_PATH)

SENSOR_COLS = ['xAcc','yAcc','zAcc','xGyro','yGyro','zGyro']
LABEL_COL   = 'label'
LABEL_NAMES = {1: 'mild', 2: 'moderate', 3: 'high'}

assert list(df.columns) == SENSOR_COLS + [LABEL_COL], 'Unexpected columns!'
print('Shape:', df.shape)
print('\nClass distribution:')
for lbl, cnt in df[LABEL_COL].value_counts().sort_index().items():
    print(f'  {lbl} ({LABEL_NAMES[lbl]}): {cnt}')
df.head()

## 4. Visualise the three severity levels

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
colors = {1: 'seagreen', 2: 'orange', 3: 'tomato'}

for i, lbl in enumerate([1, 2, 3]):
    sub = df[df[LABEL_COL] == lbl]
    gyro = sub[['xGyro','yGyro','zGyro']].values
    var_per_axis = np.var(gyro, axis=0)
    best_axis = ['xGyro','yGyro','zGyro'][int(np.argmax(var_per_axis))]
    sig = sub[best_axis].values[:300]

    axes[i][0].plot(sig, color=colors[lbl], linewidth=0.8)
    axes[i][0].set_title(f'Label {lbl} ({LABEL_NAMES[lbl]}) - {best_axis} time domain', fontweight='bold')
    axes[i][0].set_ylabel('deg/s'); axes[i][0].grid(alpha=0.3)

    x = sig - np.mean(sig)
    N = len(x)
    freqs = fftfreq(N, d=1/FS)[:N//2]
    mag = np.abs(fft(x))[:N//2]
    axes[i][1].plot(freqs, mag, color=colors[lbl], linewidth=0.9)
    axes[i][1].axvspan(3, 6, alpha=0.12, color='red')
    axes[i][1].set_title(f'Label {lbl} - frequency domain', fontweight='bold')
    axes[i][1].set_xlabel('Hz'); axes[i][1].set_xlim(0, 25); axes[i][1].grid(alpha=0.3)

plt.suptitle('Tremor severity - red band = 3-6 Hz Parkinsonian range', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

## 5. Feature extraction (MUST match feature_adapter.py)

Same logic as `app/core/feature_adapter.py` - DC removed before FFT,
features on the most active single axis. Training and live inference must
use identical feature code or predictions will be wrong.

In [ ]:
FEATURE_NAMES = ['dominant_freq','peak_magnitude','power_0_3hz','power_3_6hz',
                 'power_6_12hz','power_12plus','ratio_parkinson','ratio_essential',
                 'mean','std','rms','max_abs','skewness','kurtosis','jerk_mean']

def extract_fft_features(signal_1d, fs=FS):
    x = np.asarray(signal_1d, dtype=np.float64)
    N = len(x)
    if N < 4:
        return {k: 0.0 for k in FEATURE_NAMES}
    x_ac = x - np.mean(x)
    freqs = fftfreq(N, d=1/fs)[:N//2]
    magnitude = np.abs(fft(x_ac))[:N//2]
    di = int(np.argmax(magnitude))
    def bp(lo, hi):
        m = (freqs >= lo) & (freqs < hi)
        return float(np.sum(magnitude[m]**2))
    p03, p36, p612, p12 = bp(0,3), bp(3,6), bp(6,12), bp(12,25)
    tot = p03 + p36 + p612 + p12 + 1e-10
    return {
        'dominant_freq'  : float(freqs[di]),
        'peak_magnitude' : float(magnitude[di]),
        'power_0_3hz'    : p03, 'power_3_6hz': p36,
        'power_6_12hz'   : p612, 'power_12plus': p12,
        'ratio_parkinson': p36/tot, 'ratio_essential': p612/tot,
        'mean': float(np.mean(x)), 'std': float(np.std(x)),
        'rms' : float(np.sqrt(np.mean(x**2))), 'max_abs': float(np.max(np.abs(x))),
        'skewness': float(pd.Series(x).skew()), 'kurtosis': float(pd.Series(x).kurt()),
        'jerk_mean': float(np.mean(np.abs(np.diff(x)))),
    }

def dominant_axis(window_df):
    acc = window_df[['xAcc','yAcc','zAcc']].values
    var = [np.var(acc[:,i] - np.mean(acc[:,i])) for i in range(3)]
    return acc[:, int(np.argmax(var))]

print('Feature functions defined. Feature count:', len(FEATURE_NAMES))

## 6. Window each class block into feature rows

In [ ]:
WINDOW = 100        # 2 seconds at 50 Hz
STRIDE = 25         # 75% overlap -> more training samples

feature_rows, labels = [], []

for lbl in [1, 2, 3]:
    block = df[df[LABEL_COL] == lbl].reset_index(drop=True)
    for start in range(0, len(block) - WINDOW + 1, STRIDE):
        win = block.iloc[start:start + WINDOW]
        axis = dominant_axis(win)
        feats = extract_fft_features(axis)
        feature_rows.append([feats[k] for k in FEATURE_NAMES])
        labels.append(lbl)

X = np.array(feature_rows)
y_raw = np.array(labels)

print(f'Windowed feature matrix: {X.shape}')
print('Windows per class:')
for lbl in [1,2,3]:
    print(f'  {lbl} ({LABEL_NAMES[lbl]}): {np.sum(y_raw==lbl)}')

## 7. Train SVM + Random Forest, pick the winner

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y_raw)   # 1,2,3 -> 0,1,2

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True,
          class_weight='balanced', random_state=42)
svm.fit(X_tr_s, y_tr)
svm_pred = svm.predict(X_te_s)
svm_acc = accuracy_score(y_te, svm_pred)

rf = RandomForestClassifier(n_estimators=200, max_depth=12,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
rf_pred = rf.predict(X_te)
rf_acc = accuracy_score(y_te, rf_pred)

target_names = [LABEL_NAMES[c] for c in le.classes_]
print('-- SVM --'); print(classification_report(y_te, svm_pred, target_names=target_names))
print('-- Random Forest --'); print(classification_report(y_te, rf_pred, target_names=target_names))
print(f'SVM acc: {svm_acc:.4f}  |  RF acc: {rf_acc:.4f}')

BEST_NAME = 'SVM' if svm_acc >= rf_acc else 'RandomForest'
BEST = svm if svm_acc >= rf_acc else rf
best_pred = svm_pred if BEST_NAME == 'SVM' else rf_pred
print(f'-> Best: {BEST_NAME}')

In [ ]:
cm = confusion_matrix(y_te, best_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=target_names, yticklabels=target_names)
plt.title(f'{BEST_NAME} - tremor severity', fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(MODELS_DIR + 'tremor_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

cv = cross_val_score(
    SVC(kernel='rbf', C=10, gamma='scale', probability=True, class_weight='balanced'),
    scaler.transform(X), y,
    cv=StratifiedKFold(5, shuffle=True, random_state=42), n_jobs=-1)
print(f'SVM 5-fold CV: {cv.mean():.4f} +/- {cv.std():.4f}')

## 8. Save model + scaler + config

In [ ]:
with open(MODELS_DIR + 'tremor_classifier.pkl', 'wb') as f: pickle.dump(BEST, f)
with open(MODELS_DIR + 'tremor_scaler.pkl', 'wb') as f: pickle.dump(scaler, f)
with open(MODELS_DIR + 'tremor_label_encoder.pkl', 'wb') as f: pickle.dump(le, f)

config = {
    'model_file'   : 'tremor_classifier.pkl',
    'scaler_file'  : 'tremor_scaler.pkl',
    'encoder_file' : 'tremor_label_encoder.pkl',
    'model_type'   : BEST_NAME,
    'feature_names': FEATURE_NAMES,
    'class_ids'    : [int(c) for c in le.classes_],
    'class_names'  : [LABEL_NAMES[c] for c in le.classes_],
    'sampling_rate_hz': FS,
    'window_size'  : WINDOW,
    'feature_axis' : 'dominant_accel_axis',
    'cv_accuracy'  : round(float(cv.mean()), 4),
    'test_accuracy': round(float(accuracy_score(y_te, best_pred)), 4),
    'description'  : 'Tremor severity (mild/moderate/high) SVM on FFT features'
}
with open(MODELS_DIR + 'tremor_model_config.json', 'w') as f: json.dump(config, f, indent=2)

print('Saved. Config:')
for k, v in config.items(): print(f'  {k:18s}: {v}')

## 9. Inference test (mirrors the live app)

In [ ]:
clf = pickle.load(open(MODELS_DIR + 'tremor_classifier.pkl','rb'))
scl = pickle.load(open(MODELS_DIR + 'tremor_scaler.pkl','rb'))
le2 = pickle.load(open(MODELS_DIR + 'tremor_label_encoder.pkl','rb'))
cfg = json.load(open(MODELS_DIR + 'tremor_model_config.json'))

def predict_tremor_window(window_df):
    axis = dominant_axis(window_df)
    feats = extract_fft_features(axis)
    vec = scl.transform(np.array([feats[k] for k in cfg['feature_names']]).reshape(1,-1))
    idx = clf.predict(vec)[0]
    proba = clf.predict_proba(vec)[0]
    cls_id = le2.classes_[idx]
    return {'severity': LABEL_NAMES[cls_id], 'class_id': int(cls_id),
            'confidence': round(float(proba.max()),4)}

print('Test on real windows from each class:')
for lbl in [1,2,3]:
    block = df[df[LABEL_COL]==lbl].reset_index(drop=True)
    win = block.iloc[:WINDOW]
    r = predict_tremor_window(win)
    ok = 'OK ' if r['class_id']==lbl else 'XX '
    print(f'  {ok} actual={LABEL_NAMES[lbl]:9s} -> predicted={r["severity"]:9s} (conf {r["confidence"]})')
print('\nTremor model ready and matches the live feature adapter.')